[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/superconductivity/superconductivity.ipynb)

# The Superconducting Higgs Mode

This example computes the collective amplitude oscillation of a superconductor after a sudden drop in pairing attraction, which is called the superconducting Higgs mode. A conventional superconductor pairs electrons of opposite momentum and spin across the energy band, so each pair of orbitals sits empty, filled, or in a coherent superposition of both. All pairs contribute their coherence to one shared superconducting gap, and that gap drives every pair in turn, causing the disturbed gap amplitude to ring at twice its asymptotic value as [Tsuji and Aoki](https://arxiv.org/abs/1404.2711) and [Chou, Liao and Foster](https://arxiv.org/abs/1611.07089) analyse for ultrafast quench experiments.

Each pair orbital in internal pairing space is modelled as a pseudospin, represented as the intrinsic patch of the plane it turns in so that `xy & spin` reads pair occupation and `yz & spin` reads pairing coherence. The effective field is an extrinsic bivector combining single-particle dispersion in `xy` with the collective gap in `yz` and `zx`, turning every pseudospin through `(-2 * field).commutator(spins)` while `field & spin` pairs each field with its pseudospin in the reduced BCS energy. The term ***duality map*** here means a linear map from an intrinsic quantity to an extrinsic one of the complementary grade, and the pairing interaction is a symmetric duality map sending the band-averaged pseudospin coherence into the shared gap. Linearizing the commutator about equilibrium separates local precession at fixed gap from collective gap feedback across the band.

In the vector notation of the Anderson pseudospin equations, both the pair state $\mathbf s_k = (s_k^x, s_k^y, s_k^z)$ and the effective field $\mathbf b_k = (-\Delta_x, -\Delta_y, \xi_k)$ are written as three-component vectors in a fictitious $\mathbb R^3$, precessing via the cross product $\dot{\mathbf s}_k = 2\,\mathbf b_k\times\mathbf s_k$ with transverse projection $\boldsymbol\Delta = g\,N^{-1}\sum_k (s_k^x, s_k^y, 0)$ and reduced BCS energy $H = \sum_k 2\xi_k s_k^z - g\,N\,|\langle\mathbf s_\perp\rangle|^2$. Linearizing around equilibrium $\mathbf s_{k0}$ expands the cross product into two skew-symmetric matrix terms, $\delta\dot{\mathbf s}_k = 2\,[\mathbf b_{k0}]_\times\,\delta\mathbf s_k - 2\,[\mathbf s_{k0}]_\times\,g\langle\delta\mathbf s_\perp\rangle$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from functools import partial

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_states
from examples.quantum.superconductivity import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
# + extrinsic, - intrinsic
# A pseudospin is the intrinsic patch of the plane it turns in, paired with an extrinsic driving field.
Spin = ga.gatype.antibivector()                             # Antibivector-
Field = ga.gatype.bivector()                                # Bivector+
Pairing = ga.gatype((Field, Spin))                          # Field+ <- Spin-
SpinMap = ga.gatype((Spin, Spin))                           # Spin- <- Spin-

## Ground-state pairing across the band

At zero temperature, each pseudospin aligns opposite to its effective field as a patch of length one half square to the field's plane, `-field.dual().normalized() / 2`. Because that effective field includes the collective gap generated by all levels together, repeated alignment iterates to the self-consistent ground state across an equally weighted energy band measured in one reference energy unit with Planck's reduced constant set to one.

In [ ]:
n_levels = 240
cutoff = 5.0
coupling = 4.4
n_alignment_iterations = 160
seed_gap = mv.yz                                            # [] Field+

In [ ]:
# Pairing coherence in yz and zx produces a gap field in those same planes, while occupation in xy does not pair.
coherence = mv.yz * (mv.yz & Spin) + mv.zx * (mv.zx & Spin)  # [] Field+ <- Spin-
pairing = coupling * coherence                              # [] Field+ <- Spin-


def energy_levels(n_levels: int, cutoff: float) -> Field:
    """Midpoint quadrature places `n_levels` equally weighted orbitals across `[-cutoff, cutoff]` (`Field+[levels]`)."""
    energies = cutoff * (2 * (np.arange(n_levels) + 0.5) / n_levels - 1)
    return (mv.xy * energies).field()                       # Field+[levels]


def ground_state(dispersion: Field, pairing: Pairing, seed_gap: Field, n_iterations: int) -> Spin:
    """Repeated alignment sets each length-1/2 pseudospin square against the self-consistent field (`Spin-[levels]`)."""
    gap = seed_gap                                          # [] Field+
    for _ in range(n_iterations):
        spins = -(dispersion - gap).dual().normalized() / 2  # Spin-[levels]
        gap = pairing(spins.sites.mean())                   # [] Field+
    return spins


dispersion = energy_levels(n_levels, cutoff)                # Field+[levels]
initial_spins = ground_state(dispersion, pairing, seed_gap, n_alignment_iterations)  # Spin-[levels]
save_figure(render.equilibrium_figure(dispersion, initial_spins), "superconductivity_equilibrium");

Across the chemical potential, electron occupation drops smoothly from filled states below the Fermi level to empty states above it, while pairing coherence peaks right at the Fermi level where empty and filled pair states mix equally.

## Pairing-strength quench

When the pairing attraction drops suddenly, the pseudospins keep their initial orientations while the gap they generate shrinks immediately with the weaker coupling. Each pseudospin is then misaligned with its new effective field and begins precessing at an energy-dependent rate while feeding its changing coherence back into the shared gap at every step.

In [ ]:
quench_ratio = 0.88
dt = 0.04
n_steps = 600
n_steps_per_frame = 2
n_midpoint_iterations = 3
frame_duration_ms = 40

In [ ]:
post_pairing = quench_ratio * pairing                       # [] Field+ <- Spin-
n_frames = n_steps // n_steps_per_frame + 1
times = np.arange(n_frames) * n_steps_per_frame * dt


def precession_rate(spins: Spin, dispersion: Field, pairing: Pairing) -> Spin:
    """Each pseudospin precesses in the effective field whose gap comes from the band-averaged spin (`Spin-[levels]`)."""
    gap = pairing(spins.sites.mean())                       # [] Field+
    field = dispersion - gap                                # Field+[levels]
    return (-2 * field).commutator(spins)                   # Spin-[levels]


def evolution(
    state: Spin, rate: Callable[[Spin], Spin], dt: float,
    n_steps: int, n_steps_per_frame: int, n_iterations: int,
) -> Iterator[Spin]:
    """Yields the initial state and every `n_steps_per_frame`-th implicit midpoint update (`[...] Spin-[levels]`)."""
    yield state
    for _ in range(n_steps // n_steps_per_frame):
        for _ in range(n_steps_per_frame):
            # Fixed-point iteration evaluates the rate halfway across the step, preserving quadratic invariants.
            end = state
            for _ in range(n_iterations):
                end = state + dt * rate((state + end) / 2)
            state = end
        yield state


quench_rate = partial(precession_rate, dispersion=dispersion, pairing=post_pairing)
quench_states = show_states(
    ((spins, post_pairing(spins.sites.mean())) for spins in evolution(
        initial_spins, quench_rate, dt, n_steps, n_steps_per_frame, n_midpoint_iterations,
    )),
    render.quench_animation, dispersion, times,
    name="superconductivity_quench", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Arrows coloured from blue below the Fermi level to red above it show the pairing coherence in the `yz` and `zx` planes for sampled orbitals, while the teal curve on the right tracks the collective gap magnitude across all levels as the outer orbitals wind progressively faster than the modes near zero energy.

## Gap oscillations after the quench

In an isolated collisionless condensate, the post-quench BCS energy stays constant while orbitals at different energies dephase from one another. Because that conserved energy exceeds the ground-state energy at the reduced coupling, the collective gap rings at twice its asymptotic value and settles below the new zero-temperature equilibrium gap.

In [ ]:
def bcs_energy(spins: Spin, dispersion: Field, pairing: Pairing) -> Scalar:
    """Pairs each orbital's dispersion with its pseudospin and subtracts the shared pairing interaction once (`[...] Scalar+`)."""
    mean_spin = spins.sites.mean()                          # [...] Spin-
    return 2 * (dispersion & spins).sites.mean() - (pairing(mean_spin) & mean_spin)


quench_history = stack([spins for spins, _ in quench_states])  # [n_frames] Spin-[levels]
gaps = stack([gap for _, gap in quench_states])                # [n_frames] Field+
final_equilibrium = ground_state(dispersion, post_pairing, seed_gap, n_alignment_iterations)  # Spin-[levels]
final_gap = post_pairing(final_equilibrium.sites.mean())       # [] Field+
save_figure(render.gap_figure(times, gaps, final_gap), "superconductivity_gap");

The solid teal curve shows the isolated condensate's gap ringing down by dephasing, while the dashed dark line marks the larger ground-state gap at the reduced attraction.

## Collective linear response

A weak quench disturbs both the pseudospin being turned and the collective gap field doing the turning. Evaluating the commutator on the disturbance space `Spin` separates local precession in the equilibrium field from collective feedback of the disturbed mean spin across all levels.

In [ ]:
small_quench_ratio = 0.99
feedback_strengths = np.array([1.0, 0.0])
response_names = ("collective response", "fixed pairing field")

In [ ]:
weak_pairing = small_quench_ratio * pairing                 # [] Field+ <- Spin-
weak_equilibrium = ground_state(dispersion, weak_pairing, seed_gap, n_alignment_iterations)  # Spin-[levels]
weak_gap = weak_pairing(weak_equilibrium.sites.mean())      # [] Field+


def response_maps(dispersion: Field, pairing: Pairing, equilibrium: Spin) -> tuple[SpinMap, SpinMap]:
    """Linearizing about `equilibrium` gives the local precession map and the collective feedback map (`(Spin- <- Spin-)[levels]`)."""
    field = dispersion - pairing(equilibrium.sites.mean())  # Field+[levels]
    # Varying a disturbance turns the disturbed spin locally and shifts the shared pairing field across all levels.
    local_map = (-2 * field).commutator(Spin)               # (Spin- <- Spin-)[levels]
    feedback_map = (2 * pairing).commutator(equilibrium)    # (Spin- <- Spin-)[levels]
    return local_map, feedback_map


local_map, feedback_map = response_maps(dispersion, weak_pairing, weak_equilibrium)
feedback_maps = feedback_map * feedback_strengths           # [2] (Spin- <- Spin-)[levels]
disturbance = (initial_spins - weak_equilibrium).broadcast_to(feedback_strengths.shape)  # [2] Spin-[levels]
linear_history = stack(evolution(
    disturbance, lambda delta: local_map(delta) + feedback_maps(delta.sites.mean()),
    dt, n_steps, n_steps_per_frame, n_midpoint_iterations,
))                                                          # [n_frames, 2] Spin-[levels]
weak_history = stack(evolution(
    initial_spins, partial(precession_rate, dispersion=dispersion, pairing=weak_pairing),
    dt, n_steps, n_steps_per_frame, n_midpoint_iterations,
))                                                          # [n_frames] Spin-[levels]
exact_gap_change = weak_pairing(weak_history.sites.mean()) - weak_gap   # [n_frames] Field+
predicted_gap_change = weak_pairing(linear_history.sites.mean())        # [n_frames, 2] Field+
save_figure(render.response_figure(times, exact_gap_change, predicted_gap_change, response_names), "superconductivity_response");

The dashed teal curve with collective feedback stays on top of the broad nonlinear trace throughout the ringdown, whereas the dashed orange curve with a fixed pairing field drifts immediately in both amplitude and frequency.

In [ ]:
# checks
initial_gap = pairing(initial_spins.sites.mean())
assert ((initial_spins + (dispersion - initial_gap).dual().normalized() / 2).antinorm() < 1e-12).all()

energies = bcs_energy(quench_history, dispersion, post_pairing)
assert ((energies - energies[0]).abs() < 1e-4).all()
assert bcs_energy(final_equilibrium, dispersion, post_pairing) < energies[0]

occupations = (mv.xy & quench_history).sites.mean()
assert ((occupations - occupations[0]).abs() < 1e-12).all()
assert ((quench_history.antinorm() - 0.5).abs() < 1e-3).all()

assert ((exact_gap_change - predicted_gap_change[:, 0]).norm() < 1e-4).all()
assert ((exact_gap_change - predicted_gap_change[:, 1]).norm() > 5e-3).any()

phase_shift = -mv.xy.commutator(weak_equilibrium)
assert ((local_map(phase_shift) + feedback_map(phase_shift.sites.mean())).antinorm() < 1e-12).all()